### 판례 목록 수집

In [1]:
import requests
import time
import pandas as pd

# 1. 설정 정보
USER_ID = "silverbreak"  # 필수 입력
BASE_URL = "http://www.law.go.kr/DRF/lawSearch.do"

def collect_cases_by_org(org_code, org_name):
    """
    특정 법원 코드(org_code)에 대한 판례 목록을 끝까지 수집합니다.
    """
    all_cases = []
    page = 1
    
    print(f"--- [{org_name}] 수집 시작 ---")
    
    while True:
        params = {
            "OC": USER_ID,
            "target": "prec",
            "type": "JSON",       # JSON 형식 지정
            "org": org_code,      # 법원 종류
            "display": 100,       # 한 번에 최대 100개
            "page": page          # 페이지 번호
        }
        
        try:
            response = requests.get(BASE_URL, params=params)
            
            # 응답 코드가 200이 아니면 중단
            if response.status_code != 200:
                print(f"Error: 페이지 {page} 요청 실패 (Status: {response.status_code})")
                break
                
            data = response.json()
            
            # 검색 결과 확인 (API 구조상 'Prec' 키에 리스트가 담김)
            # 데이터가 없거나 에러인 경우 구조가 다를 수 있음
            search_result = data.get("PrecSearch", {})
            cases = search_result.get("prec", [])
            
            # 1. 데이터가 아예 없는 경우 (빈 리스트 or None) -> 수집 종료
            if not cases:
                print(f"[{org_name}] {page}페이지: 데이터 없음. 수집 종료.")
                break
            
            # 2. JSON 파싱 특이사항 처리: 결과가 1개일 때 리스트가 아니라 딕셔너리로 오는 경우 방지
            if isinstance(cases, dict):
                cases = [cases]
            
            # 데이터 저장
            all_cases.extend(cases)
            print(f"[{org_name}] {page}페이지 수집 완료 ({len(cases)}건)")
            
            # 3. 가져온 개수가 display(100)보다 적으면 마지막 페이지임 -> 수집 종료
            # (이 조건이 없어도 다음 루프에서 empty 체크로 걸러지지만, 효율을 위해 추가)
            if len(cases) < 100:
                print(f"[{org_name}] 마지막 페이지 도달.")
                break
                
            page += 1
            time.sleep(0.3)  # 서버 부하 방지용 딜레이
            
        except Exception as e:
            print(f"Error 발생: {e}")
            break
            
    print(f"--- [{org_name}] 총 {len(all_cases)}건 수집 완료 ---\n")
    return all_cases

# --- 실행부 ---
if __name__ == "__main__":
    # 대법원(400201) 수집
    supreme_court_cases = collect_cases_by_org("400201", "대법원")
    
    # 하위법원(400202) 수집
    lower_court_cases = collect_cases_by_org("400202", "하위법원")

--- [대법원] 수집 시작 ---
[대법원] 1페이지 수집 완료 (100건)
[대법원] 2페이지 수집 완료 (100건)
[대법원] 3페이지 수집 완료 (100건)
[대법원] 4페이지 수집 완료 (100건)
[대법원] 5페이지 수집 완료 (100건)
[대법원] 6페이지 수집 완료 (100건)
[대법원] 7페이지 수집 완료 (100건)
[대법원] 8페이지 수집 완료 (100건)
[대법원] 9페이지 수집 완료 (100건)
[대법원] 10페이지 수집 완료 (100건)
[대법원] 11페이지 수집 완료 (100건)
[대법원] 12페이지 수집 완료 (100건)
[대법원] 13페이지 수집 완료 (100건)
[대법원] 14페이지 수집 완료 (100건)
[대법원] 15페이지 수집 완료 (100건)
[대법원] 16페이지 수집 완료 (100건)
[대법원] 17페이지 수집 완료 (100건)
[대법원] 18페이지 수집 완료 (100건)
[대법원] 19페이지 수집 완료 (100건)
[대법원] 20페이지 수집 완료 (100건)
[대법원] 21페이지 수집 완료 (100건)
[대법원] 22페이지 수집 완료 (100건)
[대법원] 23페이지 수집 완료 (100건)
[대법원] 24페이지 수집 완료 (100건)
[대법원] 25페이지 수집 완료 (100건)
[대법원] 26페이지 수집 완료 (100건)
[대법원] 27페이지 수집 완료 (100건)
[대법원] 28페이지 수집 완료 (100건)
[대법원] 29페이지 수집 완료 (100건)
[대법원] 30페이지 수집 완료 (100건)
[대법원] 31페이지 수집 완료 (100건)
[대법원] 32페이지 수집 완료 (100건)
[대법원] 33페이지 수집 완료 (100건)
[대법원] 34페이지 수집 완료 (100건)
[대법원] 35페이지 수집 완료 (100건)
[대법원] 36페이지 수집 완료 (100건)
[대법원] 37페이지 수집 완료 (100건)
[대법원] 38페이지 수집 완료 (100건)
[대법원] 39페이지 수집 완료 (100건)
[대법원] 40페이지 수집

In [3]:
import pandas as pd
import os

# --- 데이터 저장부 ---

# 1. 대법원 판례 저장
if supreme_court_cases:
    df_supreme = pd.DataFrame(supreme_court_cases)
    # 파일명: supreme_court_list.csv
    df_supreme.to_csv("대법원_판례_리스트.csv", index=False, encoding="utf-8-sig")
    print(f"✅ [대법원] 총 {len(df_supreme)}건 저장 완료 (파일명: 대법원_판례_리스트.csv)")
else:
    print("⚠️ [대법원] 수집된 데이터가 없어 파일을 저장하지 않았습니다.")

# 2. 하위법원 판례 저장
if lower_court_cases:
    df_lower = pd.DataFrame(lower_court_cases)
    # 파일명: lower_court_list.csv
    df_lower.to_csv("하위법원_판례_리스트.csv", index=False, encoding="utf-8-sig")
    print(f"✅ [하위법원] 총 {len(df_lower)}건 저장 완료 (파일명: 하위법원_판례_list.csv)")
else:
    print("⚠️ [하위법원] 수집된 데이터가 없어 파일을 저장하지 않았습니다.")

# (선택) 저장된 파일 경로 확인
print(f"\n파일 저장 위치: {os.getcwd()}")

✅ [대법원] 총 7100건 저장 완료 (파일명: 대법원_판례_리스트.csv)
✅ [하위법원] 총 93721건 저장 완료 (파일명: 하위법원_판례_list.csv)

파일 저장 위치: c:\Users\Administrator\Documents\GitHub\Ko-legal-stopwords\data


### 대법원 데이터 수집

In [1]:
import pandas as pd
import requests
import time
import os
import json
import re

# 1. 설정 정보
INPUT_FILE = "대법원_판례_리스트.csv"   # 판례 목록 파일
SAVE_DIR = "대법원_판례"                # 저장할 폴더명
BASE_URL_HOST = "http://www.law.go.kr"

# 2. 저장 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f"폴더 생성 완료: {SAVE_DIR}")

# 3. 파일 로드
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE)
    print(f"목록 파일 로드 완료: {len(df)}건")
else:
    print(f"오류: '{INPUT_FILE}' 파일이 없습니다.")
    exit()

# 헤더 설정
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

def sanitize_filename(name):
    """파일명에 쓸 수 없는 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

print(f"--- 개별 판례 수집 시작 ---")

# 4. 순회 및 저장
for idx, row in df.iterrows():
    try:
        # 파일명으로 쓸 사건번호 가져오기
        case_no = row.get('사건번호')
        link_part = row.get('판례상세링크')

        # 필수 정보 없으면 패스
        if pd.isna(case_no) or pd.isna(link_part):
            continue

        # 파일명 정제 (사건번호에 혹시 모를 특수문자 제거)
        safe_name = sanitize_filename(case_no)
        file_path = os.path.join(SAVE_DIR, f"{safe_name}.json")

        # [이어하기 기능] 이미 수집된 파일이면 건너뛰기
        if os.path.exists(file_path):
            # print(f"[{idx+1}] 이미 존재함: {safe_name}")
            continue

        # URL 생성 (HTML -> JSON)
        link_part_json = link_part.replace("type=HTML", "type=JSON")
        full_url = f"{BASE_URL_HOST}{link_part_json}"

        # API 요청
        response = requests.get(full_url, headers=headers, timeout=15)

        if response.status_code == 200:
            data = response.json()
            
            # JSON 구조 파싱
            prec_service = data.get("PrecService", {})
            if "Prec" in prec_service:
                target_info = prec_service["Prec"]
            else:
                target_info = prec_service
            
            # 데이터 유효성 확인 및 저장
            if target_info and ("판례일련번호" in target_info or "판례정보일련번호" in target_info):
                
                # JSON 파일로 저장 (한글 깨짐 방지: ensure_ascii=False)
                with open(file_path, 'w', encoding='utf-8') as f:
                    json.dump(target_info, f, ensure_ascii=False, indent=4)
                
                # 진행 상황 출력 (50건마다)
                if (idx + 1) % 50 == 0:
                    print(f"[{idx+1}/{len(df)}] 저장 완료: {safe_name}")
            else:
                print(f"[{idx+1}] 내용 없음 확인: {full_url}")
        else:
            print(f"[{idx+1}] 요청 실패 ({response.status_code}): {safe_name}")

        # 서버 부하 방지
        time.sleep(0.2)

    except Exception as e:
        print(f"[{idx+1}] 에러 발생: {e}")

print("\n작업이 완료되었습니다.")

폴더 생성 완료: 대법원_판례
목록 파일 로드 완료: 7100건
--- 개별 판례 수집 시작 ---
[1] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=608741&type=JSON&mobileYn=
[2] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=608757&type=JSON&mobileYn=
[3] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=609047&type=JSON&mobileYn=
[4] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=608781&type=JSON&mobileYn=
[7] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=608687&type=JSON&mobileYn=
[11] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=608177&type=JSON&mobileYn=
[16] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=608701&type=JSON&mobileYn=
[18] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=prec&ID=608761&type=JSON&mobileYn=
[20] 내용 없음 확인: http://www.law.go.kr/DRF/lawService.do?OC=silv

In [1]:
import json
import os
import re  # 정규표현식 (HTML 태그 처리를 위해 사용)

# 1. 설정 정보
SOURCE_DIR = '대법원_판례'       # 원본 JSON 파일이 있는 폴더
TARGET_DIR = '대법원_판례_txt화'  # 결과 텍스트가 저장될 폴더

def convert_json_to_txt(source_folder, target_folder):
    # 폴더 생성
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"결과 저장 폴더 생성: {target_folder}")

    if not os.path.exists(source_folder):
        print(f"오류: 원본 폴더 '{source_folder}'가 없습니다.")
        return

    # JSON 파일 목록 가져오기
    file_list = [f for f in os.listdir(source_folder) if f.endswith('.json')]
    print(f"총 {len(file_list)}개의 판례 파일 변환을 시작합니다.")

    for i, file_name in enumerate(file_list):
        try:
            source_path = os.path.join(source_folder, file_name)
            target_file_name = os.path.splitext(file_name)[0] + '.txt'
            target_path = os.path.join(target_folder, target_file_name)
            
            # 1. JSON 파일 읽기
            with open(source_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 2. "판례내용" 추출
            # 해당 키가 없을 경우 빈 문자열 반환
            content = data.get('판례내용', '')
            
            if not content:
                print(f"주의: {file_name}에 '판례내용'이 비어있거나 없습니다.")
                
            # 3. 전처리 (HTML 태그 제거 및 줄바꿈 처리)
            # <br/>, <br>, <br /> 등을 실제 줄바꿈(\n)으로 변경
            content = re.sub(r'<br\s*/?>', '\n', content)
            
            # (선택) 연속된 공백을 하나로 줄이거나, 앞뒤 공백 제거
            content = content.strip()

            # 4. txt 파일로 저장
            with open(target_path, 'w', encoding='utf-8') as f:
                f.write(content)
            
            if (i + 1) % 100 == 0:
                print(f"[{i+1}/{len(file_list)}] 변환 완료...")

        except json.JSONDecodeError:
            print(f"Error: {file_name} 파일은 올바른 JSON 형식이 아닙니다.")
        except Exception as e:
            print(f"Error processing file {file_name}: {e}")
            continue

    print(f"\n모든 작업이 완료되었습니다. 결과 경로: {target_folder}")

if __name__ == "__main__":
    convert_json_to_txt(SOURCE_DIR, TARGET_DIR)

결과 저장 폴더 생성: 대법원_판례_txt화
총 5472개의 판례 파일 변환을 시작합니다.
[100/5472] 변환 완료...
[200/5472] 변환 완료...
[300/5472] 변환 완료...
[400/5472] 변환 완료...
[500/5472] 변환 완료...
[600/5472] 변환 완료...
[700/5472] 변환 완료...
[800/5472] 변환 완료...
[900/5472] 변환 완료...
[1000/5472] 변환 완료...
[1100/5472] 변환 완료...
[1200/5472] 변환 완료...
[1300/5472] 변환 완료...
[1400/5472] 변환 완료...
[1500/5472] 변환 완료...
[1600/5472] 변환 완료...
[1700/5472] 변환 완료...
[1800/5472] 변환 완료...
[1900/5472] 변환 완료...
[2000/5472] 변환 완료...
[2100/5472] 변환 완료...
[2200/5472] 변환 완료...
[2300/5472] 변환 완료...
[2400/5472] 변환 완료...
[2500/5472] 변환 완료...
[2600/5472] 변환 완료...
[2700/5472] 변환 완료...
[2800/5472] 변환 완료...
[2900/5472] 변환 완료...
[3000/5472] 변환 완료...
[3100/5472] 변환 완료...
[3200/5472] 변환 완료...
[3300/5472] 변환 완료...
[3400/5472] 변환 완료...
[3500/5472] 변환 완료...
[3600/5472] 변환 완료...
[3700/5472] 변환 완료...
[3800/5472] 변환 완료...
[3900/5472] 변환 완료...
[4000/5472] 변환 완료...
[4100/5472] 변환 완료...
[4200/5472] 변환 완료...
[4300/5472] 변환 완료...
[4400/5472] 변환 완료...
[4500/5472] 변환 완료...
[4600/5472] 변

In [ ]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import os
import json
import re

# 1. 설정 정보
INPUT_FILE = "하위법원_판례_리스트.csv"
SAVE_DIR = "하위법원_판례"
BASE_URL_HOST = "http://www.law.go.kr"

# 2. 저장 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 3. 파일명 정제 함수 (미리 정의)
def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

# 4. 데이터 로드 및 전처리
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE)
    print(f"전체 목록 로드: {len(df)}건")
    
    # (1) 대법원 필터링
    if '데이터출처명' in df.columns:
        df = df[df['데이터출처명'] == '대법원'].copy() # copy()로 경고 방지
        print(f"-> '대법원' 필터링 후: {len(df)}건")
    
    # (2) [핵심] 이미 저장된 파일 목록 확인 (확장자 제거)
    # 폴더 내의 .json 파일들의 이름(사건번호)을 미리 집합(set)으로 만듭니다.
    downloaded_files = {f.replace('.json', '') for f in os.listdir(SAVE_DIR) if f.endswith('.json')}
    print(f"-> 현재 폴더에 저장된 파일: {len(downloaded_files)}건")

    # (3) DataFrame에 '파일명' 컬럼 미리 생성 (비교를 위해)
    # 사건번호를 파일명 규칙대로 변환해서 새로운 열을 만듭니다.
    df['safe_name'] = df['사건번호'].apply(sanitize_filename)

    # (4) 이미 다운로드된 건 제외 (차집합)
    # safe_name이 downloaded_files에 없는 행만 남깁니다.
    target_df = df[~df['safe_name'].isin(downloaded_files)]
    
    remaining_count = len(target_df)
    print(f"===> [최종 다운로드 대상]: {remaining_count}건 (이미 완료된 {len(df) - remaining_count}건 제외)")

    if remaining_count == 0:
        print("모든 파일이 이미 다운로드되었습니다. 작업을 종료합니다.")
        exit()

else:
    print(f"오류: '{INPUT_FILE}' 파일이 없습니다.")
    exit()

# --- 세션 설정 (재시도 로직) ---
session = requests.Session()
retry = Retry(connect=3, backoff_factor=0.5)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
})

print(f"\n--- 남은 {remaining_count}건 수집 시작 ---")

# 5. 순회 및 저장 (target_df만 순회하므로 이미 받은 건 건너뜀)
# total 변수는 진행률 표시용
for count, (idx, row) in enumerate(target_df.iterrows(), start=1):
    try:
        case_no = row.get('사건번호')
        link_part = row.get('판례상세링크')
        safe_name = row.get('safe_name') # 위에서 만들어둔 컬럼 사용
        
        file_path = os.path.join(SAVE_DIR, f"{safe_name}.json")
        
        # URL 생성
        link_part_json = link_part.replace("type=HTML", "type=JSON")
        full_url = f"{BASE_URL_HOST}{link_part_json}"

        # 요청 및 에러 핸들링
        try:
            response = session.get(full_url, timeout=20)
            
            if response.status_code == 200:
                data = response.json()
                prec_service = data.get("PrecService", {})
                target_info = prec_service.get("Prec", prec_service)
                
                if target_info and ("판례일련번호" in target_info or "판례정보일련번호" in target_info):
                    with open(file_path, 'w', encoding='utf-8') as f:
                        json.dump(target_info, f, ensure_ascii=False, indent=4)
                    
                    # 진행 상황 출력 (현재 / 남은 건수)
                    if count % 10 == 0: # 10건마다 출력으로 변경 (확인 자주 하도록)
                        print(f"[{count}/{remaining_count}] 저장 완료: {safe_name}")
                else:
                    print(f"[{count}] 내용 없음: {full_url}")
            else:
                print(f"[{count}] 요청 실패 ({response.status_code}): {safe_name}")

            # 대기 시간 (1.2초로 약간 더 여유 있게)
            time.sleep(1.2)

        except requests.exceptions.RequestException as e:
            print(f"\n!!! [{count}/{remaining_count}] 차단/네트워크 에러: {e}")
            print("!!! 60초 대기 후 재시도합니다...")
            time.sleep(60)
            # 여기서는 continue하지 않고 다음 루프로 넘어가게 둡니다. 
            # (엄밀한 재시도 로직을 짜면 복잡해지니, 일단 1분 쉬고 다음 건으로 넘어갑니다)

    except Exception as e:
        print(f"[{count}] 일반 에러: {e}")

print("\n작업이 완료되었습니다.")

전체 목록 파일 로드 완료: 93721건
-> '대법원' 필터링 후 대상: 27826건
--- 개별 판례 수집 시작 ---
[650/27826] 저장 완료: 2022라717
[700/27826] 저장 완료: 2022허3229
[750/27826] 저장 완료: 2022누1550
[800/27826] 저장 완료: 2022노3872
[850/27826] 저장 완료: 2022고합45
[900/27826] 저장 완료: 2020나2000160
[950/27826] 저장 완료: 2022노189
[1000/27826] 저장 완료: 2021구합8898
[1050/27826] 저장 완료: 2021나218176
[1100/27826] 저장 완료: 2021노490
[1150/27826] 저장 완료: 2021누5025
[1200/27826] 저장 완료: 2021나2000327
[1250/27826] 저장 완료: 2021나12672
[1300/27826] 저장 완료: 2021가단16005
[1350/27826] 저장 완료: 2021누31032
[1400/27826] 저장 완료: 2021누10796
[1450/27826] 저장 완료: 2021나2009911
[1500/27826] 저장 완료: 2018가합52481
[1550/27826] 저장 완료: 2019가합110538
[1600/27826] 저장 완료: 2020나15660
[1650/27826] 저장 완료: 2020나48927
[1700/27826] 저장 완료: 2020가소16553
[1750/27826] 저장 완료: 2020나41990
[1800/27826] 저장 완료: 2020구합51198
[1850/27826] 저장 완료: 2019구합107097
[1900/27826] 저장 완료: 2019나64103
[1950/27826] 저장 완료: 2019노869
[2000/27826] 저장 완료: 2020고단8570
[2050/27826] 저장 완료: 2019누5565
[2100/27826] 저장 완료: 2019나12568
[2150/27